# Local OCR & Data Extraction Pipeline

This notebook implements and demonstrates the document processing pipeline. It aligns scanned PDFs with their respective metadata CSVs, runs OCR (RapidOCR/EasyOCR) on the first page, extracts entities (movement type, names, office numbers), and builds a unified Parquet dataset.

---

In [1]:
# 1. Imports and environment setup
import os
import re
import csv
import pandas as pd
import numpy as np
import cv2
import fitz
from rapidocr_onnxruntime import RapidOCR

### 2. Alignment Logic
We define a robust mapping function to match PDFs to their CSV metadata files. We use three alignment methods:
1. **Filename matching**: Direct substring overlap within the same directory.
2. **Office number in filename**: Regex matching the office number in the PDF filename to the CSV `MOTIVO` field.
3. **Office number in OCR text**: Matching the office number extracted from the first page text to the CSV.

In [2]:
def clean_name(name):
    name = name.lower()
    name = re.sub(r"^(10_|11_|12_|13_|14_|15_|16_|17_|18_|19_|20_|21_|22_|23_|24_|25_|1_|2_|3_|4_|5_|6_|7_|8_|9_)", "", name)
    name = re.sub(r"(_cnbv|_vs|\.pdf|\.csv)", "", name)
    name = name.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u").replace("ñ", "n")
    return re.sub(r"[\_\-\s]+", "", name)

def extract_office_number(text):
    # Matches office format: 110/K/2924/2026 or 110-G-1329-2026
    m = re.search(r"110[/\-\s_]?[GKgk][/\-\s_]?\d+[/\-\s_]?\d+", text)
    if m:
        return re.sub(r"[/\-\s_]+", "_", m.group(0).lower())
    return None

### 3. Rule-Based Extractor (Regex)
We implement a parser to extract details directly from the OCR text of the first page to evaluate how well simple rules perform compared to machine learning. We target:
- **Movement type**: `ALTA` (added to list) vs `BAJA` (removed/suspended from list).
- **Office number**: Matches the pattern `110/K/...` or `110/G/...`.
- **Entity names**: Matches uppercase words following keywords like "Se elimina... a:" or "Lista de Personas Bloqueadas a:".

In [3]:
def rule_based_extract(text):
    clean_txt = " ".join(text.split())
    
    # 1. Extract Movement
    movement = "UNKNOWN"
    # Baja keywords
    if any(kw in clean_txt.lower() for kw in ["elimina provisionalmente", "suspension definitiva", "dejar sin efecto"]):
        movement = "BAJA"
    # Alta keywords
    elif any(kw in clean_txt.lower() for kw in ["actualizacion de la lista", "personas bloqueadas", "adicionar"]):
        # Double check it is not a Baja referencing an Alta
        if not any(kw in clean_txt.lower() for kw in ["elimina provisionalmente", "suspension definitiva"]):
            movement = "ALTA"
            
    # 2. Extract Office Number (Motivo)
    office = ""
    m_off = re.search(r"Oficio\s*No\.?\s*(110[/\-\s]?[GKgk][/\-\s]?\d+[/\-\s]?\d+)", clean_txt, re.IGNORECASE)
    if m_off:
        office = "OFICIO NO. " + m_off.group(1).upper()
        
    return {
        "extracted_movement": movement,
        "extracted_office": office
    }

### 4. Load and Validate the Generated Parquet Dataset
The background script `build_dataset.py` processes all PDFs and CSVs and outputs `files_dataset.parquet`. Let's load it and inspect its properties.

In [4]:
# Load the dataset
parquet_path = "files_dataset.parquet"
if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print(f"Parquet dataset loaded successfully! Shape: {df.shape}")
    
    # Value counts
    print("\nMovement Label Distribution:")
    print(df["label_movement"].value_counts())
    
    # Sample records
    display(df[["pdf_name", "label_movement", "label_nombre", "match_type"]].head(5))
else:
    print("files_dataset.parquet not found yet. Make sure build_dataset.py has finished running.")

Parquet dataset loaded successfully! Shape: (115, 14)

Movement Label Distribution:
label_movement
ALTA    94
BAJA    21
Name: count, dtype: int64


,pdf_name,label_movement,label_nombre,match_type
0,tps_rental_cnbv.pdf,BAJA,TPS RENTAL SA DE CV,filename_match
1,everardo_rojas_contreras_cnbv.pdf,BAJA,EVERARDO,filename_match
2,dany_noe_altamirano_diaz_cnbv.pdf,BAJA,DANY NOE,filename_match
3,jesus_humberto_ramos_mejia.pdf,BAJA,JESUS HUMBERTO,filename_match
4,11_juan_antonio_luquin_mendoza.pdf,ALTA,JUAN ANTONIO,filename_match


### 5. Evaluate Name and Rule-Based Extractions
Let's run our rule-based parser on all documents and evaluate its accuracy against the ground truth labels.

In [5]:
if os.path.exists(parquet_path):
    parsed_results = []
    for idx, row in df.iterrows():
        extracted = rule_based_extract(row["extracted_text"])
        parsed_results.append(extracted)
        
    df_parsed = pd.DataFrame(parsed_results)
    df_eval = pd.concat([df, df_parsed], axis=1)
    
    # 1. Evaluate Rule-based Movement Classification
    correct_movement = (df_eval["label_movement"] == df_eval["extracted_movement"]).sum()
    accuracy_movement = (correct_movement / len(df_eval)) * 100
    print(f"Rule-Based Movement Classification Accuracy: {accuracy_movement:.2f}% ({correct_movement}/{len(df_eval)})")
    
    # 2. Evaluate Office Number Extraction
    # Normalize spaces and compare
    def norm_motivo(val):
        return re.sub(r"[^a-zA-Z0-9]", "", str(val).lower())
    
    correct_office = df_eval.apply(lambda r: norm_motivo(r["label_motivo"]) in norm_motivo(r["extracted_office"]), axis=1).sum()
    accuracy_office = (correct_office / len(df_eval)) * 100
    print(f"Rule-Based Office Number Extraction Accuracy: {accuracy_office:.2f}% ({correct_office}/{len(df_eval)})")
else:
    print("Parquet dataset not loaded.")

Rule-Based Movement Classification Accuracy: 83.48% (96/115)


Rule-Based Office Number Extraction Accuracy: 99.13% (114/115)
